# 🚨 Lesson 2 Practical: Suspicious Login Investigation

**Estimated time:** 20 minutes  
**Level:** First-time Jupyter users with Security+ knowledge and some basic Python/PCEP experience

## Scenario

It is **08:15 on Monday morning**. Your Security Operations Centre has received several authentication alerts from overnight activity.

You have been given a small login dataset. Your job is to use the same toolkit introduced in Lesson 2 to:

- inspect security data with **Pandas**
- identify suspicious behaviour
- build a simple **risk score**
- visualise the results with **Matplotlib**
- decide what should actually be investigated
- explain why **human context still matters**

> **Important:** This is a guided investigation. You do not need to know Jupyter already. Every command is explained before you run it.

---

## How to use this notebook

A Jupyter notebook is made of **cells**.

- **Markdown cells** contain instructions like this one.
- **Code cells** contain Python.
- Click a code cell and press **Ctrl + Enter** to run it.
- The output will appear directly underneath the cell.
- Run the notebook **from top to bottom** because later cells use information created by earlier cells.

If you make a mistake, that is fine. Read the error or feedback, correct the value, and run the cell again.


## Step 0 — Prepare the investigation environment
**Time: about 2 minutes**

Before analysing anything, we need to load the Python libraries we will use.

### Why are we running these commands?

- `import pandas as pd` loads **Pandas**, which helps us organise and analyse data in tables.
- `import matplotlib.pyplot as plt` loads **Matplotlib**, which helps us turn data into graphs.
- We also create a tiny feedback function so the notebook can tell you **why an answer is correct or incorrect**.

Run the cell below. You only need to run it once.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def check_answer(name, your_answer, correct_answer, correct_message, incorrect_message):
    if str(your_answer).strip().lower() == str(correct_answer).strip().lower():
        print(f"✅ Correct, {name}!")
        print(correct_message)
    else:
        print(f"❌ Not quite, {name}.")
        print(incorrect_message)
        print("Change your answer and run this cell again.")

print("✅ Investigation tools loaded successfully.")

## Step 1 — Load the security data
**Time: about 2 minutes**

In a real SOC, this information might come from Microsoft Entra ID, Active Directory, a SIEM, VPN logs, or another authentication system.

For this activity, the dataset is built directly into the notebook so you do not need to download any files.

Each row represents one user's recent login behaviour.

### Columns

- `username` — account name
- `country` — country the login came from
- `login_hour` — hour of the login using 24-hour time
- `failed_attempts` — number of failed login attempts
- `device_count` — number of different devices observed
- `mfa_failed` — whether MFA failed
- `new_country` — whether the login came from a country not normally used by that account

Run the cell below to create the dataset.


In [ ]:
data = {
    "username": ["Alice", "Bob", "Eve", "Musa", "Sarah", "Daniel", "Lebo", "Priya"],
    "country": ["South Africa", "South Africa", "Russia", "South Africa",
                "Germany", "South Africa", "South Africa", "India"],
    "login_hour": [9, 14, 3, 8, 2, 11, 22, 10],
    "failed_attempts": [1, 2, 34, 3, 12, 0, 7, 1],
    "device_count": [2, 1, 9, 2, 5, 1, 3, 2],
    "mfa_failed": ["No", "No", "Yes", "No", "Yes", "No", "No", "No"],
    "new_country": ["No", "No", "Yes", "No", "Yes", "No", "No", "No"]
}

df = pd.DataFrame(data)

print("✅ Dataset created.")
df

### What just happened?

`pd.DataFrame(data)` converted our Python dictionary into a **DataFrame**.

A DataFrame is similar to a spreadsheet: it has **rows and columns**, but Python can search, filter, sort and calculate across it very quickly.

This is useful in cybersecurity because real logs may contain **thousands or millions of rows**. We do not want to manually inspect every one.


## Step 2 — Inspect the evidence
**Time: about 3 minutes**

Before making decisions, an analyst should first understand the data.

Run the next command:

```python
df.head()
```

### Why?

`head()` shows the first few rows of a DataFrame. It is a quick way to confirm that the data loaded correctly and to understand what the columns look like.


In [ ]:
df.head()

### Quick Check 1

Look at the table above.

Which user has the **highest number of failed login attempts**?

Change only the text inside the quotation marks below, then run the cell.

Example:

```python
answer_1 = "Alice"
```


In [ ]:
answer_1 = "TYPE YOUR ANSWER HERE"

check_answer(
    "analyst",
    answer_1,
    "Eve",
    "Eve has 34 failed attempts, far more than any other user. A high number of failures can indicate password guessing, brute-force activity, or a user repeatedly entering the wrong password. It is suspicious, but it is not proof of an attack by itself.",
    "Look again at the 'failed_attempts' column. Find the largest number, then identify the username on that same row."
)

## Step 3 — Let Python sort the evidence
**Time: about 2 minutes**

Manually scanning a small table is easy. Real SOC data is much larger.

Run:

```python
df.sort_values("failed_attempts", ascending=False)
```

### Why?

- `sort_values(...)` tells Pandas to sort the table using a specific column.
- `"failed_attempts"` is the column we want to sort by.
- `ascending=False` means **highest to lowest**.

This lets an analyst immediately bring the most interesting records to the top.


In [ ]:
df.sort_values("failed_attempts", ascending=False)

### Quick Check 2

After sorting the data, which **two users** appear at the top?

Enter them in order, separated by a comma.

Example:

```python
answer_2 = "Alice, Bob"
```


In [ ]:
answer_2 = "TYPE YOUR ANSWER HERE"

check_answer(
    "analyst",
    answer_2,
    "Eve, Sarah",
    "Correct. Eve and Sarah have the two highest failed-login counts. This does not mean both accounts are compromised. It means they deserve closer inspection.",
    "The table is sorted from the highest failed-attempt count to the lowest. Read the first two usernames."
)

## Step 4 — Build a simple risk score
**Time: about 4 minutes**

Security tools often combine several signals instead of looking at only one event.

For this activity, we will use these simple rules:

| Security signal | Risk points |
|---|---:|
| More than 10 failed attempts | +30 |
| More than 4 devices | +20 |
| MFA failed | +30 |
| Login before 05:00 | +20 |
| Login from a new country | +20 |

> This is a **teaching model**, not a real production detection model.

### Why do this?

One strange login time might be harmless. One MFA failure might also be harmless. But **several suspicious signals together** can create a stronger reason to investigate.

Run the code below.


In [ ]:
df["risk_score"] = 0

df.loc[df["failed_attempts"] > 10, "risk_score"] += 30
df.loc[df["device_count"] > 4, "risk_score"] += 20
df.loc[df["mfa_failed"] == "Yes", "risk_score"] += 30
df.loc[df["login_hour"] < 5, "risk_score"] += 20
df.loc[df["new_country"] == "Yes", "risk_score"] += 20

df[["username", "failed_attempts", "device_count", "mfa_failed", "new_country", "login_hour", "risk_score"]]

### What do these commands mean?

Take this line:

```python
df.loc[df["failed_attempts"] > 10, "risk_score"] += 30
```

Read it like this:

> **Find every row where failed attempts are greater than 10, then add 30 points to that row's risk score.**

The other lines do the same thing for different security signals.

This is similar to the risk-score example from the lesson: several signals are converted into numbers so the most concerning activity can be prioritised.


### Quick Check 3

Which user now has the **highest risk score**?

Enter the username below.


In [ ]:
answer_3 = "TYPE YOUR ANSWER HERE"

check_answer(
    "analyst",
    answer_3,
    "Eve",
    "Correct. Eve triggers every risk rule: many failed attempts, many devices, failed MFA, an early-hours login, and a new country. The score is high because multiple suspicious signals appear together.",
    "Look at the 'risk_score' column and find the largest value. Remember: the score combines several security signals."
)

## Step 5 — Filter the accounts that need attention
**Time: about 2 minutes**

A SOC analyst usually does not want to investigate every event.

For this exercise, we will say that any account with a risk score of **50 or more** should be reviewed.

Run:

```python
suspicious = df[df["risk_score"] >= 50]
```

### Why?

This creates a new DataFrame containing only rows where the risk score meets our investigation threshold.

This is called **filtering**.


In [ ]:
suspicious = df[df["risk_score"] >= 50]

suspicious[["username", "country", "risk_score"]]

### Quick Check 4

How many accounts meet the investigation threshold of **50 or more**?

Enter only the number.


In [ ]:
answer_4 = 0  # replace 0 with your answer

check_answer(
    "analyst",
    answer_4,
    2,
    "Correct. Two accounts currently meet the threshold: Eve and Sarah. The filter has reduced the full dataset to a much smaller investigation queue.",
    "Count the rows displayed in the filtered 'suspicious' table. Do not count the header."
)

## Step 6 — Make the risk visible
**Time: about 2 minutes**

Tables are useful, but graphs can make patterns easier to spot.

Run the code below to create a bar chart.

### Why?

- `plt.bar(...)` creates the bars.
- The usernames go on the horizontal axis.
- Risk scores go on the vertical axis.
- `plt.show()` displays the chart.

A SOC dashboard often uses visualisations because a large spike or outlier can be easier to notice visually than inside a large table.


In [ ]:
df_sorted = df.sort_values("risk_score", ascending=False)

plt.figure(figsize=(9, 4))
plt.bar(df_sorted["username"], df_sorted["risk_score"])
plt.title("Login Risk Score by User")
plt.xlabel("User")
plt.ylabel("Risk Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Quick Check 5

The chart should make two users stand out clearly.

Which statement is the best interpretation?

**A.** Every account with a score above zero is definitely compromised.  
**B.** Eve and Sarah should be investigated because they have the highest combined risk signals.  
**C.** The graph proves Eve installed malware.  
**D.** The graph proves the scoring system is always correct.

Enter only **A, B, C, or D**.


In [ ]:
answer_5 = "TYPE A, B, C OR D"

check_answer(
    "analyst",
    answer_5,
    "B",
    "Correct. The chart helps us prioritise Eve and Sarah because several risk signals combine to make them stand out. It does not prove compromise or explain the full context.",
    "Remember the lesson: AI and automated scoring can recommend what deserves attention, but a human analyst still needs evidence and context. Look for the option that says 'investigate', not 'definitely compromised'."
)

## Step 7 — New intelligence arrives
**Time: about 2 minutes**

🚨 **UPDATE FROM HR**

HR confirms that **Sarah is currently attending a conference in Germany**.

She also contacted the help desk last night because she replaced her phone and was having trouble with MFA.

### Why does this matter?

Our scoring system correctly noticed that Sarah's behaviour was **unusual**:

- new country
- failed MFA
- unusual time
- multiple devices

But unusual does not automatically mean malicious.

This is where **human context** becomes important.


### Final Decision

Based on the new information, what is the best action for Sarah?

**A.** Immediately disable Sarah's account because the AI score is high.  
**B.** Ignore the alert completely because HR says she is travelling.  
**C.** Verify the travel/help-desk information and monitor the account, while prioritising Eve for immediate investigation.  
**D.** Delete Sarah's login records from the dataset.

Enter only **A, B, C, or D**.


In [ ]:
answer_6 = "TYPE A, B, C OR D"

check_answer(
    "analyst",
    answer_6,
    "C",
    "Correct. Sarah's activity still deserves verification, but the new context provides a reasonable explanation. Eve has the stronger unexplained pattern and should be prioritised. This demonstrates why automated detection supports human analysts rather than replacing them.",
    "Do not treat the risk score as proof. Security decisions should combine technical signals with verified business context. Choose the option that verifies Sarah's explanation while still prioritising the stronger unexplained threat."
)

# 📝 Final SOC Mini Report
**Time: about 1 minute**

You have completed the technical investigation.

In a real SOC, findings must be communicated clearly. Replace the placeholder text below with your own short answers.

You do **not** need to write a long report.


In [ ]:
most_suspicious_account = "TYPE USERNAME"
main_evidence = "TYPE 1-2 SHORT SENTENCES"
recommended_action = "TYPE 1 SHORT SENTENCE"

print("SOC MINI REPORT")
print("----------------")
print("Most suspicious account:", most_suspicious_account)
print("Evidence:", main_evidence)
print("Recommended action:", recommended_action)

## ✅ Activity Complete

You have just used a simplified version of an AI-enabled cybersecurity workflow:

**Security data → Pandas → filtering → risk scoring → visualisation → human investigation**

### What you practised

- running cells in **Jupyter**
- reading a **Pandas DataFrame**
- sorting and filtering security data
- converting security signals into a numerical **risk score**
- creating a basic **Matplotlib** visualisation
- distinguishing **suspicious** from **confirmed malicious**
- using human context before making a security decision

### Key lesson

> **The tool can tell you what looks unusual. The analyst still has to decide what it means.**

That is the same principle we use when working with more advanced machine-learning and AI security systems.
